In [0]:
%sql

-- ============================================================
-- GOLD INITIAL FULL LOAD
-- Silver → Gold
-- ============================================================


-- ============================================================
-- 1. DIM ASSET
-- ============================================================

INSERT INTO motordatabricks.utilities_gold.dim_asset
SELECT DISTINCT
    xxhash64(asset_id) AS asset_key,
    asset_id,
    asset_type,
    manufacturer,
    model,
    serial_number,
    installation_date,
    location_id,
    city,
    region,
    criticality,
    asset_status
FROM motordatabricks.utilities_silver.assets
WHERE asset_id IS NOT NULL;


-- ============================================================
-- 2. DIM TECHNICIAN
-- ============================================================

INSERT INTO motordatabricks.utilities_gold.dim_technician
SELECT DISTINCT
    xxhash64(technician_id) AS technician_key,
    technician_id,
    technician_name,
    skill_level,
    team,
    experience_years
FROM motordatabricks.utilities_silver.technicians
WHERE technician_id IS NOT NULL;


-- ============================================================
-- 3. DIM LOCATION
-- ============================================================

INSERT INTO motordatabricks.utilities_gold.dim_location
SELECT DISTINCT
    xxhash64(location_id) AS location_key,
    location_id,
    city,
    region
FROM motordatabricks.utilities_silver.locations
WHERE location_id IS NOT NULL;


-- ============================================================
-- 4. DIM DATE
-- ============================================================

INSERT INTO motordatabricks.utilities_gold.dim_date

WITH date_range AS (

    SELECT
        LEAST(
            MIN(TO_DATE(created_date)),
            MIN(TO_DATE(scheduled_date)),
            MIN(TO_DATE(completed_date))
        ) AS min_date,

        GREATEST(
            MAX(TO_DATE(created_date)),
            MAX(TO_DATE(scheduled_date)),
            MAX(TO_DATE(completed_date))
        ) AS max_date

    FROM motordatabricks.utilities_silver.maintenance
)

SELECT
    CAST(DATE_FORMAT(date, 'yyyyMMdd') AS INT) AS date_key,
    date,
    YEAR(date) AS year,
    QUARTER(date) AS quarter,
    MONTH(date) AS month,
    DATE_FORMAT(date, 'MMMM') AS month_name,
    WEEKOFYEAR(date) AS week,
    DAY(date) AS day,
    DAYOFWEEK(date) AS day_of_week,
    DATE_FORMAT(date, 'EEEE') AS day_name,

    CASE
        WHEN DAYOFWEEK(date) IN (1, 7)
        THEN TRUE
        ELSE FALSE
    END AS is_weekend

FROM date_range

LATERAL VIEW EXPLODE(
    SEQUENCE(
        min_date,
        max_date,
        INTERVAL 1 DAY
    )
) dates AS date

WHERE min_date IS NOT NULL
  AND max_date IS NOT NULL;


-- ============================================================
-- 5. DIM MAINTENANCE TYPE
-- ============================================================

INSERT INTO motordatabricks.utilities_gold.dim_maintenance_type
SELECT DISTINCT

    xxhash64(
        maintenance_type,
        maintenance_category,
        planned_flag,
        failure_type,
        failure_severity,
        priority
    ) AS maintenance_type_key,

    maintenance_type,
    maintenance_category,
    planned_flag,
    failure_type,
    failure_severity,
    priority

FROM motordatabricks.utilities_silver.maintenance;


-- ============================================================
-- 6. DIM STATUS
-- ============================================================

INSERT INTO motordatabricks.utilities_gold.dim_status
SELECT DISTINCT

    xxhash64(
        work_order_status,
        resolution_code
    ) AS status_key,

    work_order_status,
    resolution_code

FROM motordatabricks.utilities_silver.maintenance;


-- ============================================================
-- 7. FACT MAINTENANCE
-- ============================================================

INSERT INTO motordatabricks.utilities_gold.fact_maintenance

SELECT

    -- Business / Fact Key
    xxhash64(work_order_id) AS maintenance_key,

    work_order_id,

    -- Dimension Keys
    xxhash64(asset_id) AS asset_key,

    xxhash64(technician_id) AS technician_key,

    xxhash64(location_id) AS location_key,

    xxhash64(
        maintenance_type,
        maintenance_category,
        planned_flag,
        failure_type,
        failure_severity,
        priority
    ) AS maintenance_type_key,

    xxhash64(
        work_order_status,
        resolution_code
    ) AS status_key,

    -- Date Keys
    CAST(
        DATE_FORMAT(
            TO_DATE(created_date),
            'yyyyMMdd'
        ) AS INT
    ) AS created_date_key,

    CAST(
        DATE_FORMAT(
            TO_DATE(scheduled_date),
            'yyyyMMdd'
        ) AS INT
    ) AS scheduled_date_key,

    CAST(
        DATE_FORMAT(
            TO_DATE(completed_date),
            'yyyyMMdd'
        ) AS INT
    ) AS completed_date_key,

    -- Measures
    downtime_hours,
    labor_hours,
    parts_used,
    parts_cost,
    labor_cost,
    total_cost,

    -- Additional attributes
    created_date,
    scheduled_date,
    completed_date,
    source_system,
    updated_at

FROM motordatabricks.utilities_silver.maintenance;

num_affected_rows,num_inserted_rows
3600,3600


In [0]:
%sql

SELECT 'dim_asset' AS table_name, COUNT(*) AS row_count
FROM motordatabricks.utilities_gold.dim_asset

UNION ALL

SELECT 'dim_technician', COUNT(*)
FROM motordatabricks.utilities_gold.dim_technician

UNION ALL

SELECT 'dim_location', COUNT(*)
FROM motordatabricks.utilities_gold.dim_location

UNION ALL

SELECT 'dim_date', COUNT(*)
FROM motordatabricks.utilities_gold.dim_date

UNION ALL

SELECT 'dim_maintenance_type', COUNT(*)
FROM motordatabricks.utilities_gold.dim_maintenance_type

UNION ALL

SELECT 'dim_status', COUNT(*)
FROM motordatabricks.utilities_gold.dim_status

UNION ALL

SELECT 'fact_maintenance', COUNT(*)
FROM motordatabricks.utilities_gold.fact_maintenance;

table_name,row_count
dim_asset,500
dim_technician,100
dim_location,72
dim_date,188
dim_maintenance_type,660
dim_status,11
fact_maintenance,3600
